In [ ]:
import pypsa
import pandas as pd
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
from _helpers import mock_snakemake
snakemake = mock_snakemake(
            "collect_figures",
            scenario="mga-nwa-iso", #"", # penalty-sa, penalty-ea, penalty-nwa, penalty-oc
            sort="cost_average",
            demand=False,
    )

In [ ]:
scenario = snakemake.wildcards.scenario

### Paths

In [ ]:
trade_model = f"../../results/cost_year~2050/interone~hbi/intertwo~eaf-grid/final~steel/scenario~{scenario}/network.nc"
# single_region = "../../resources/lcos/cost_year~2030/Europe/network_1.nc" # old
single_region = "../../resources/lco-steel/cost_year~2030/South_South_America/network_1.nc"

### Read trade model

In [ ]:
n = pypsa.Network(trade_model)

In [ ]:
n_ssa = pypsa.Network(single_region)

### Iron ore supply costs (w/o transport)

In [ ]:
iron_ore_cost = n.generators[n.generators.carrier == "iron_ore"].marginal_cost.mean()
iron_ore_to_steel = 1/n.links[n.links.carrier == 'steel'].efficiency.mean()
iron_ore_cost_notransport = iron_ore_cost * iron_ore_to_steel
print(f"Iron ore cost (w/o transport): {iron_ore_cost_notransport:.2f} €/t_steel")

Insight: iron ore costs are allocated in the transport model, as generator for each region. They are not included in the fundamental steel model, since iron ore trade is separate from steel modelling

### Steel supply costs

In [ ]:
# Steel supply cost distribution per region (boxplot), including iron ore cost w/o transport
steel_links = n.links[n.links.carrier == 'steel'].copy()
steel_links['region'] = steel_links['bus1'].str.replace('_ore$', '', regex=True)
# Add iron ore cost w/o transport to each steel supply cost. The marginal cost of steel links refers to bus0, so an adjustment to steel (bus1) is required
steel_links['total_cost'] = steel_links['marginal_cost'] * iron_ore_to_steel + iron_ore_cost_notransport

fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(x='region', y='total_cost', data=steel_links, ax=ax, width=0.5, color='steelblue', fliersize=3)
ax.set_ylabel('Steel supply cost incl. iron ore (€/t_product)')
ax.set_xlabel('Region')
ax.set_title('Steel supply cost distribution per region (incl. iron ore material cost)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.set_ylim(0, None)
ax.grid(axis='y')
plt.tight_layout()
plt.show()

### Steel supply costs single region

In [ ]:
costs_per_carrier = n_ssa.statistics.system_cost() / n_ssa.statistics.energy_balance().loc["Load","Steel","Steel"]
costs_per_carrier #.sum()

In [ ]:
# Stacked bar chart of costs per carrier for single region
costs = costs_per_carrier.copy() * (-1)
costs = costs.groupby(level="carrier").sum()

fig, ax = plt.subplots(figsize=(6, 4))
colors = sns.color_palette('Set2', len(costs))
ax.bar(['Total'], [costs.sum()], color='lightgrey', label='Total')
bottom = 0
for i, (carrier, value) in enumerate(zip(costs.index, costs.values)):
    ax.bar(['Total'], [value], bottom=bottom, color=colors[i], label=carrier)
    bottom += value
ax.set_ylabel('Cost (€/t_steel)')
ax.set_xlabel('Carrier')
ax.set_title('Steel supply costs single region (w/o iron ore material cost)')
ax.grid(axis='y', alpha=0.4, zorder=0)
ax.legend(title='Carrier', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_ylim(0,500)
plt.tight_layout()
plt.show()

In [ ]:
# Sanity check: objective value divided by total steel load should equal cost per ton steel
n_ssa.objective / n_ssa.statistics.energy_balance().loc["Load","Steel","Steel"] * (-1)

### Iron ore and steel price

In [ ]:
# Iron ore price analysis
iron_ore_price = n.buses_t.marginal_price.loc[:, n.buses[n.buses.carrier == "iron_ore"].index]

# Steel price analysis
steel_price = n.buses_t.marginal_price.loc[:, n.buses[n.buses.carrier == "steel"].index]

In [ ]:
# Helper to clean region names (remove _ore)
def clean_region_names(index):
    return [re.sub(r'_ore$', '', str(i)) for i in index]

# Prepare data for both carriers
data = {
    'iron_ore': iron_ore_price,
    'steel': steel_price
}
titles = {
    'iron_ore': 'Iron Ore',
    'steel': 'Steel'
}

fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for i, (carrier, price_df) in enumerate(data.items()):
    # Clean region names
    price_df = price_df.copy()
    price_df.columns = clean_region_names(price_df.columns)
    price_T = price_df.T
    price_T.columns = ['Price']
    price_T.index = clean_region_names(price_T.index)

    # Boxplot (narrower)
    sns.boxplot(data=price_df.melt(var_name='Region', value_name=""), ax=axes[i,0], width=0.3)
    axes[i,0].set_title(f'{titles[carrier]} Price Distribution Across Regions')
    axes[i,0].set_ylabel('Price (€/t_product)')
    axes[i,0].set_xlabel('Region')
    axes[i,0].set_ylim(0, price_df.max().max() * 1.1)

    # Bar plot
    price_T.plot(kind='bar', ax=axes[i,1], legend=False, width=0.5)
    axes[i,1].set_title(f'{titles[carrier]} Price per Region')
    axes[i,1].set_ylabel('Price (€/t_product)')
    axes[i,1].set_xlabel('Region')
    axes[i,1].set_xticklabels(price_T.index, rotation=90)
    axes[i,1].set_ylim(0, price_df.max().max() * 1.1)
    axes[i,1].grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.show()

### Prices in MGA runs

In [ ]:
n.statistics()

In [ ]:
n.objective / 1e6

In [ ]:
n.buses_t.marginal_price.loc[:, n.buses[n.buses.carrier == "steel"].index]

### Cost and prices

Thesis: The price of steel equals the cost of a single region + transport costs of iron ore and steel. The cost of a single region benchmark depends to what degree the steel making potential in the respective region is used ("marginal"). Hence, if all potentials are used, even the most expensive ones, then this is the benchmark.

steel price = steel production (region, marginal) + transport iron ore + transport steel